We'll split all the data into chunks , keeping the content and metadata seperate so that each chunk retains information about it's source . 

In [1]:
import os
from langchain_core.documents import Document

data_dir = "../data/raw"
documents = []

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".md"):
        continue

    with open(os.path.join(data_dir, file_name), encoding="utf-8") as f:
        lines = f.read().splitlines()

    title = lines[0].replace("#", "").strip()
    source = lines[1].replace("Source", "").lstrip(" :").strip()
    content = "\n".join(lines[2:]).strip()

    doc = Document(
        page_content=content,
        metadata={"title": title, "source": source, "file": file_name}
    )
    documents.append(doc)

print("Documents loaded:", len(documents))
for d in documents:
    print(d.metadata)


Documents loaded: 5
{'title': 'Manage billing in Notion', 'source': 'https://www.notion.com/help/billing', 'file': 'billing.md'}
{'title': 'Change your Notion plan', 'source': 'https://www.notion.com/help/upgrade-or-downgrade-your-plan', 'file': 'change-your-plan.md'}
{'title': 'Intro to databases', 'source': 'https://www.notion.com/help/create-a-database', 'file': 'database-basics.md'}
{'title': 'What is Notion AI?', 'source': 'https://www.notion.com/help/notion-ai-faqs', 'file': 'notion-ai-faqs.md'}
{'title': 'Sharing & permissions settings in Notion', 'source': 'https://www.notion.com/en-gb/help/sharing-and-permissions', 'file': 'sharing-and-permissions.md'}


Splitting into chunks 

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

print("\n--- First chunk with metadata ---")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)


c:\Users\sahil\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Total chunks: 144

--- First chunk with metadata ---
Learn how you're billed for Notion, and how you can change your billing interval.

You always pay for Notion at the start of your billing interval.

If you upgrade your plan in the middle of your billing interval, the new plan will take effect immediately and start a new billing period. You'll be charged immediately for the new plan. The amount due will be reduced by a prorated amount based on the time remaining in your previous billing period for your current plan.

Metadata: {'title': 'Manage billing in Notion', 'source': 'https://www.notion.com/help/billing', 'file': 'billing.md'}


Building a Embedding model 

In [3]:
# pip install langchain-huggingface sentence-transformers faiss-cpu


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

sample_vector = embedding_model.embed_query("How do I share a page with a guest?")
print(" A vector length:", len(sample_vector))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 A vector length: 384


<!-- pip install langchain-community -->

In [11]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_model)
vector_store.save_local("../data/faiss_index")

print(" Total chunks in  Vector store :", vector_store.index.ntotal)



C:\Users\sahil\AppData\Local\Temp\ipykernel_26164\3411748375.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


 Total chunks in  Vector store : 144


Testing

In [12]:
results = vector_store.similarity_search("How do I share a page with a guest?", k=3)

for r in results:
    print("\nSOURCE:", r.metadata["title"])
    print(r.page_content[:200])



SOURCE: Sharing & permissions settings in Notion
1. Select Share at the top of any page.
2. Find and select the names of the members or groups in your workspace that you want to share the page with. You can also invite someone who isn't in your work

SOURCE: Sharing & permissions settings in Notion
This request will go to the page creator, who can accept or ignore it. You'll be notified once your request has been accepted — try refreshing the page if you're not able to edit it.

If you're a work

SOURCE: Sharing & permissions settings in Notion
You can manage general access to your page by selecting Share at the top of a page and configuring General access. Here, you can choose to share your page with:
